![Demo workflow](/Volumes/ws_databricks/default/sharedfiles/Demo workflow diagram.png)


## Step 1: Set Up the Catalog Structure.

Creating the top-level namespace structure: one `catalog` and three `schemas` following the **medallion architecture** naming pattern



```txt

Unity Catalog
└── dev  [catalog]
    │   tags: environment=development | data_classification=internal
    │
    ├── bronze  [schema]
    │   └── raw_files/  [managed volume]
    │       └── customer.csv
    │       └── sales.csv
    │
    ├── silver  [schema]
    │   ├── dim_customer  [table]   
    │   ├── fact_sales  [table]
    │   ├── vw_student_enrollments  [view]
    │   │   └── joins students + courses + enrollments
    │   └── get_grade_classification(grade)  [function]
    │       └── returns STRING  (A / B / C / D / F)
    │
    └── gold  [schema]
        └── vw_department_enrollment_stats  [materialized view]
            └── aggregates enrollments + courses
                (department, total_enrollments, avg_grade, distinct_students)
```

In [0]:
%sql
      

CREATE CATALOG IF NOT EXISTS IDENTIFIER(:catalog_name) MANAGED LOCATION 'abfss://unity-catalog-storage@dbstoragemnvhufhuewptq.dfs.core.windows.net/7405610468937344'
  COMMENT 'Solaris Energy analytics platform — solar farms and wind turbines across Europe';

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog_name || '.1_bronze') MANAGED LOCATION 'abfss://unity-catalog-storage@dbstoragemnvhufhuewptq.dfs.core.windows.net/7405610468937344'
  COMMENT 'Raw ingested data — unmodified, as received from field systems';

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog_name || '.2_silver') MANAGED LOCATION 'abfss://unity-catalog-storage@dbstoragemnvhufhuewptq.dfs.core.windows.net/7405610468937344'
  COMMENT 'Cleaned, validated, and enriched data';

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog_name || '.3_gold') MANAGED LOCATION 'abfss://unity-catalog-storage@dbstoragemnvhufhuewptq.dfs.core.windows.net/7405610468937344'
  COMMENT 'Aggregated, analytics-ready data';  
  
CREATE VOLUME IF NOT EXISTS IDENTIFIER(:catalog_name || '.1_bronze.raw_files')
  COMMENT 'Landing zone for raw CSV files';